# 02 - Data Cleaning

Turns the raw Excel file into a clean table in `Data/Processed/telco_churn_clean.csv`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("../Data/Raw/Telco_customer_churn.xlsx")
print("raw shape:", df.shape)

raw shape: (7043, 33)


## Column names

In [2]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df = df.rename(columns={"customerid": "customer_id"})
list(df.columns)

['customer_id', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']

## Fix data types

In [3]:
# blanks in total_charges belong to customers with tenure 0
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")
print("NaN after conversion:", df["total_charges"].isna().sum())
print("All have tenure 0:", (df.loc[df["total_charges"].isna(), "tenure_months"] == 0).all())

df["total_charges"] = df["total_charges"].fillna(0.0)

NaN after conversion: 11
All have tenure 0: True


## Drop columns with no information

In [4]:
drop_cols = ["count", "country", "state", "lat_long"]   # constant columns and a duplicate of latitude/longitude
df = df.drop(columns=drop_cols)
df.shape

(7043, 29)

## Consistent categories

In [5]:
cat_cols = [
    "gender", "senior_citizen", "partner", "dependents", "phone_service", "multiple_lines",
    "internet_service", "online_security", "online_backup", "device_protection", "tech_support",
    "streaming_tv", "streaming_movies", "contract", "paperless_billing", "payment_method", "churn_label",
]
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip()

for c in ["multiple_lines", "online_security", "streaming_tv", "internet_service"]:
    print(c, "->", sorted(df[c].unique()))

multiple_lines -> ['No', 'No phone service', 'Yes']
online_security -> ['No', 'No internet service', 'Yes']
streaming_tv -> ['No', 'No internet service', 'Yes']
internet_service -> ['DSL', 'Fiber optic', 'No']


`No phone service` and `No internet service` are separate labels in the source. They carry the same meaning as `No` for the add-on column but are kept for now, since the feature step decides how to handle them.

In [6]:
# readable payment labels for dashboards
df["payment_method"] = df["payment_method"].str.replace(" (automatic)", "", regex=False)
df["payment_method"].value_counts()

payment_method
Electronic check    2365
Mailed check        1612
Bank transfer       1544
Credit card         1522

## Derived columns used in SQL and Power BI

In [7]:
bins = [-1, 12, 24, 48, 72]
labels = ["0-12 months", "13-24 months", "25-48 months", "49+ months"]
df["tenure_group"] = pd.cut(df["tenure_months"], bins=bins, labels=labels).astype(str)

df["churn_reason"] = df["churn_reason"].fillna("Not applicable")
df["tenure_group"].value_counts().reindex(labels)

tenure_group
0-12 months     2186
13-24 months    1024
25-48 months    1594
49+ months      2239

## Validation

In [8]:
assert df["customer_id"].is_unique
assert df.isna().sum().sum() == 0
assert set(df["churn_value"].unique()) == {0, 1}
assert (df["churn_label"] == df["churn_value"].map({1: "Yes", 0: "No"})).all()
assert (df["total_charges"] >= 0).all()
print("checks passed")
print(df.shape)

checks passed
(7043, 30)


In [9]:
df.to_csv("../Data/Processed/telco_churn_clean.csv", index=False)
df.head()

,customer_id,city,zip_code,latitude,longitude,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason,tenure_group
0,3668-QPYBK,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer,0-12 months
1,9237-HQITU,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved,0-12 months
2,9305-CDSKC,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved,0-12 months
3,7892-POOKP,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved,25-48 months
4,0280-XJGEX,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer,103.70,5036.30,Yes,1,89,5340,Competitor had better devices,49+ months
